# LLM expert persona eval analysis

## 1) The data

### Import results

In [86]:
import pandas as pd

df = pd.read_json('results.jsonl', lines=True)

df.head()

,result,category,is_expert,question,correct_answer,llm_answer,asked_at
0,pass,math,True,Use divergence therem to evaluate $\iint_S \ve...,I,To evaluate the surface integral $\iint_S \vec...,2026-06-29 21:45:31+00:00
1,pass,law,True,"A buyer contracted in writing to purchase 1,00...",A,(A),2026-06-29 21:45:46+00:00
2,fail,business,True,"Suppose there are 8,000 hours in a year (actua...",G,(C),2026-06-29 21:46:02+00:00
3,pass,computer science,True,"Let a undirected graph G with edges E = {<0,1>...",H,We are given an undirected graph G with the fo...,2026-06-29 21:46:20+00:00
4,fail,math,False,Use divergence therem to evaluate $\iint_S \ve...,I,(D),2026-06-29 21:46:20+00:00


### Number of rows

In [87]:
is_expert_len = len(df[df['is_expert']==True])
print(f'Total rows: {is_expert_len}')

Total rows: 30


### Parsing failure rate

In [88]:
(df['result'] == 'parse_failure').groupby(df['is_expert']).mean() * 100

is_expert
False    0.000000
True     3.333333
Name: result, dtype: float64

### Inspect parsing failures

In [96]:
df[df['result'] == 'parse_failure']['llm_answer']

Series([], Name: llm_answer, dtype: str)

### Remove parsing failures

In [91]:
pre_filter_len = len(df)
df = df[df['result'] != 'parse_failure']
print(f'Rows removed: {pre_filter_len - len(df)}')

Rows removed: 1


## 2) Descriptive statistics

### Overall pass rate

In [92]:
(df['result'] == 'pass').groupby(df['is_expert']).mean() * 100

is_expert
False    40.000000
True     44.827586
Name: result, dtype: float64

### Pass rate by category

In [93]:
(df['result'] == 'pass').groupby([df['category'], df['is_expert']]).mean() * 100

category          is_expert
biology           False        100.000000
                  True         100.000000
business          False          0.000000
                  True           0.000000
chemistry         False          0.000000
                  True           0.000000
computer science  False         66.666667
                  True          66.666667
economics         False         50.000000
                  True          50.000000
engineering       False          0.000000
                  True           0.000000
health            False         33.333333
                  True          33.333333
history           False          0.000000
                  True           0.000000
law               False         25.000000
                  True          50.000000
math              False         66.666667
                  True         100.000000
other             False        100.000000
                  True         100.000000
philosophy        False         50.000000
      

## 3) Chi-squared test

In [94]:
from scipy.stats import chi2_contingency

table = pd.crosstab(df['is_expert'], df['result'])

chi2, p, dof, expected = chi2_contingency(table)

print(table)
print('\np-value:', p)

result     fail  pass
is_expert            
False        18    12
True         16    13

p-value: 0.911099253533275
